# Result Tables

Exports per-dataset LaTeX result tables (Table 1 in the paper).
Reads MLflow experiments and prints mean accuracy at each retention budget.

**Input:** MLflow experiments `final-3-{dataset}-25`  
**Output:** `output/results_{dataset}.txt` (LaTeX table rows)

In [1]:
import pandas as pd
import mlflow

from utils_run import get_single_result_table
from utils_plot import find_run, get_and_preprocess_run

Connected to local server. http://localhost:5000


## Configuration

In [2]:
RUN_SPECS = {"DataAttrOpt(5000, 0.03, 15, best, +b, alpha=best)": ["CDVM5k", "-", "blue"],
             "DataAttrOpt(10000, 0.03, 15, best, +b, alpha=best)": ["CDVM10k", "--", "blue"],
             "_nomao_DataAttrOpt(10000, 0.1, 15, 0.2, +b, alpha=0.1)": ["CDVM-n", ":", "blue"],
             "PruningOptimization": ["InfOpt", "--", "green"],
             "DataOob(1000": ["DataOOB", "-", "firebrick"],
             "DataBanzhaf(num": ["Banzhaf", "-", "maroon"],
             "RandomEvaluator()": ["Random", "-", "brown"]
             }

## Table functions

In [3]:
def print_results_tex(dataset_name, run_name, df, print_=False):

    str_ = f"\t\t\t {run_name} & \n"

    for index, row in df.reset_index().iterrows():
        s = "&" if index < len(df) - 1 else ""
        mean = f"\\textbf{{{format(row['mean'], '.3f')}}}" if row['is_max'] else format(row['mean'], '.3f')

        str_ += f"\t\t\t\t {mean} $\pm$ {format(row['std'], '.2f')} {s} % {row['axis']} \n"

    str_ += f"\t\t\t \\\\ % {dataset_name}-{run_name}\n\n"

    if print_:
        print(str_)

    # save to file
    with open(f"output/results_{dataset_name}.txt", "a") as f:
        f.write(str_ + "\n\n")

    return str_


In [4]:
def print_table(dataset_name, print_=False):
    dfs = []
    for i, run_name in enumerate(RUN_SPECS.keys()):
        if not run_name.startswith("_"):
            run_name_short = run_name.split("_")[-1]
            df = get_and_preprocess_run(dataset_name, run_name_short)
            df["method"] = RUN_SPECS[run_name][0]
            dfs.append([df, RUN_SPECS[run_name][0]])

        elif dataset_name in run_name:
            run_name_short = run_name.split("_")[-1]
            df = get_and_preprocess_run(dataset_name, run_name_short)
            df["method"] = RUN_SPECS[run_name][0]
            dfs.append([df, RUN_SPECS[run_name][0]])

    # join all dataframes from list
    all_df = pd.concat([df for df, name in dfs], axis=0).reset_index(drop=True)

    # mark max value per axis and method
    all_df['is_max'] = all_df.groupby('axis')['mean'].transform(lambda x: x == x.max())

    str_ = f"\t\t\multicolumn{{7}}{{c}}{{\\textit{{{dataset_name.split('-')[0]}}}}}\\\\"
    str_ += "\n\t\t\\hline\n"


    for _ , name in dfs:
        df = all_df[all_df['method'] == name]
        str_ += print_results_tex(dataset_name, name, df, print_=False)

    str_ += "\n\t\t\\hline\n"

    if print_:
        print(str_)

    return str_

In [5]:
dfs = []
for i, run_name in enumerate(RUN_SPECS.keys()):
    if not run_name.startswith("_"):
        run_name_short = run_name.split("_")[-1]
        df = get_and_preprocess_run("cifar10-embeddings", run_name_short)
        df["method"] = RUN_SPECS[run_name][0]
        dfs.append([df, RUN_SPECS[run_name][0]])

# join all dataframes from list
all_df = pd.concat([df for df, name in dfs], axis=0).reset_index(drop=True)

# mark max value per axis and method
all_df['is_max'] = all_df.groupby('axis')['mean'].transform(lambda x: x == x.max())
all_df

,axis,mean,std,method,is_max
0,0.70,0.59752,0.020267,CDVM5k,True
1,0.75,0.57768,0.019448,CDVM5k,False
2,0.80,0.56504,0.026103,CDVM5k,False
3,0.85,0.55056,0.018881,CDVM5k,False
4,0.90,0.52464,0.025290,CDVM5k,False
5,0.95,0.43088,0.033287,CDVM5k,False
6,0.75,0.58616,0.024569,CDVM10k,True
7,0.80,0.57784,0.022382,CDVM10k,True
8,0.85,0.56328,0.020066,CDVM10k,True
9,0.90,0.54512,0.018904,CDVM10k,True


In [6]:
def print_all_tables():
    datasets = ["nomao", "cifar10-embeddings", "pol", "imdb-embeddings", "adult", "bbc-embeddings"]
    s = ""
    for dataset in datasets:
        s+= print_table(dataset, print_=False)

    print(s)

## Generate tables

In [7]:
print_all_tables()

		\multicolumn{7}{c}{\textit{nomao}}\\
		\hline
			 CDVM5k & 
				 0.855 $\pm$ 0.02 & % 0.7 
				 0.836 $\pm$ 0.02 & % 0.75 
				 0.839 $\pm$ 0.02 & % 0.8 
				 0.837 $\pm$ 0.02 & % 0.85 
				 0.830 $\pm$ 0.02 & % 0.9 
				 0.833 $\pm$ 0.03  % 0.95 
			 \\ % nomao-CDVM5k

			 CDVM10k & 
				 0.848 $\pm$ 0.01 & % 0.7 
				 0.838 $\pm$ 0.02 & % 0.75 
				 0.833 $\pm$ 0.02 & % 0.8 
				 0.838 $\pm$ 0.02 & % 0.85 
				 0.829 $\pm$ 0.02 & % 0.9 
				 0.818 $\pm$ 0.02  % 0.95 
			 \\ % nomao-CDVM10k

			 CDVM-n & 
				 0.872 $\pm$ 0.02 & % 0.7 
				 0.855 $\pm$ 0.02 & % 0.75 
				 0.848 $\pm$ 0.02 & % 0.8 
				 0.849 $\pm$ 0.02 & % 0.85 
				 0.842 $\pm$ 0.02 & % 0.9 
				 \textbf{0.842} $\pm$ 0.02  % 0.95 
			 \\ % nomao-CDVM-n

			 InfOpt & 
				 \textbf{0.874} $\pm$ 0.00 & % 0.7 
				 \textbf{0.866} $\pm$ 0.00 & % 0.75 
				 \textbf{0.862} $\pm$ 0.00 & % 0.8 
				 \textbf{0.860} $\pm$ 0.00 & % 0.85 
				 \textbf{0.858} $\pm$ 0.00 & % 0.9 
				 0.808 $\pm$ 0.00  % 0.95 
			 \\ % nomao-InfO